# CFEU journal extension: Notebook 2 of 3, models and evaluation

Run this after Notebook 1. It reads the feature tables from `cfeu_journal` in your Google Drive. It takes about 30–45 minutes on a standard Colab runtime.

**What it tests**
- **Home Credit, all applicants:** bureau-only vs alternative-only vs combined features, each with the same two learners (logistic regression and gradient boosting), 5-fold cross-validation repeated with 3 seeds.
- **Home Credit, thin-file applicants only** (no bureau records): the same comparison trained only on that group.
- **Berka:** loan-terms-only vs cash-flow vs combined, 5-fold cross-validation repeated with 20 seeds because the sample is small.

Every result has a 95% confidence interval, and differences between models are tested with the DeLong test.

**What to send back to Claude:** `model_results.json` and `headline_results.csv` (both download at the end). They contain only summary metrics.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, json, time
import numpy as np, pandas as pd
OUT = '/content/drive/MyDrive/cfeu_journal'
!pip -q install lightgbm pyarrow

In [ ]:
# Modelling functions (no need to edit)
# ---- Modelling and evaluation (Notebook 2) ----
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

try:
    import lightgbm as lgb
    HAVE_LGB = True
except Exception:
    from sklearn.ensemble import HistGradientBoostingClassifier
    HAVE_LGB = False


def make_model(kind, seed, small_data=False):
    if kind == "logit":
        return make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler(),
                             LogisticRegression(max_iter=2000, C=1.0))
    if HAVE_LGB:
        p = dict(n_estimators=150, learning_rate=0.05, num_leaves=7, min_child_samples=20,
                 subsample=0.8, subsample_freq=1, colsample_bytree=0.8) if small_data else \
            dict(n_estimators=400, learning_rate=0.05, num_leaves=31, min_child_samples=100,
                 subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0)
        return lgb.LGBMClassifier(random_state=seed, verbose=-1, n_jobs=-1, **p)
    return HistGradientBoostingClassifier(max_iter=150 if small_data else 400, learning_rate=0.05,
                                          max_leaf_nodes=7 if small_data else 31, random_state=seed)


def oof_predictions(X, y, kind, seeds, n_splits=5, small_data=False):
    """Out-of-fold predicted default probabilities, one column per seed."""
    P = np.zeros((len(y), len(seeds)))
    for j, s in enumerate(seeds):
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=s)
        for tr, te in skf.split(X, y):
            m = make_model(kind, s, small_data)
            m.fit(X.iloc[tr], y[tr])
            P[te, j] = m.predict_proba(X.iloc[te])[:, 1]
    return P


def ks_stat(y, p):
    return float(stats.ks_2samp(p[y == 1], p[y == 0]).statistic)


def approval_metrics(y, p, approve_rate=0.7):
    """Approve the lowest-risk share; report default rate among approved."""
    cut = np.quantile(p, approve_rate)
    ap = p <= cut
    return {"approve_rate": approve_rate, "default_rate_approved": float(y[ap].mean()),
            "default_rate_declined": float(y[~ap].mean()), "threshold": float(cut)}


def calib_bins(y, p, n_bins=10):
    q = pd.qcut(p, n_bins, labels=False, duplicates="drop")
    d = pd.DataFrame({"y": y, "p": p, "bin": q}).groupby("bin").agg(pred=("p", "mean"), obs=("y", "mean"), n=("y", "size"))
    return d.reset_index().to_dict(orient="list")


def bootstrap_auc_ci(y, p, n_boot=500, seed=0):
    rng = np.random.default_rng(seed); n = len(y); vals = []
    for _ in range(n_boot):
        i = rng.integers(0, n, n)
        if y[i].min() == y[i].max():
            continue
        vals.append(roc_auc_score(y[i], p[i]))
    return [float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))]


# Fast DeLong test (Sun & Xu 2014) for two correlated AUCs on the same cases
def _midrank(x):
    J = np.argsort(x); Z = x[J]; N = len(x); T = np.zeros(N); i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1) + 1
        i = j
    T2 = np.empty(N); T2[J] = T
    return T2


def delong_test(y, p1, p2):
    y = np.asarray(y); order = np.argsort(-y, kind="mergesort")
    y = y[order]; preds = np.vstack([p1, p2])[:, order]
    m = int(y.sum()); n = len(y) - m
    pos, neg = preds[:, :m], preds[:, m:]
    tx = np.array([_midrank(r) for r in pos]); ty = np.array([_midrank(r) for r in neg])
    tz = np.array([_midrank(r) for r in preds])
    aucs = tz[:, :m].sum(axis=1) / m / n - (m + 1.0) / 2.0 / n
    v01 = (tz[:, :m] - tx) / n; v10 = 1.0 - (tz[:, m:] - ty) / m
    cov = np.cov(v01) / m + np.cov(v10) / n
    diff = aucs[0] - aucs[1]
    se = np.sqrt(cov[0, 0] + cov[1, 1] - 2 * cov[0, 1])
    z = diff / se if se > 0 else 0.0
    return {"auc_1": float(aucs[0]), "auc_2": float(aucs[1]), "diff": float(diff),
            "z": float(z), "p_value": float(2 * stats.norm.sf(abs(z)))}


def evaluate(y, P, mask=None, n_boot=500):
    """P: n x seeds out-of-fold probabilities. Seed spread plus bootstrap CI on the seed-averaged score."""
    y = np.asarray(y)
    if mask is not None:
        y, P = y[mask], P[mask]
    per_seed = [roc_auc_score(y, P[:, j]) for j in range(P.shape[1])]
    p = P.mean(axis=1)
    return {"n": int(len(y)), "default_rate": float(y.mean()),
            "auc_mean_over_seeds": float(np.mean(per_seed)), "auc_sd_over_seeds": float(np.std(per_seed)),
            "auc": float(roc_auc_score(y, p)), "auc_ci95": bootstrap_auc_ci(y, p, n_boot),
            "pr_auc": float(average_precision_score(y, p)), "brier": float(brier_score_loss(y, p)),
            "ks": ks_stat(y, p), "approval_at_70pct": approval_metrics(y, p, 0.7),
            "calibration": calib_bins(y, p)}


def run_suite(df, y, feature_sets, seeds, subgroups, small_data=False, n_boot=500, log=print):
    """Train every learner on every feature set; evaluate overall and per subgroup; DeLong vs the first set."""
    y = np.asarray(y).astype(int)
    subgroups = {g: (None if m is None else np.asarray(m, bool)) for g, m in subgroups.items()}
    results, oof = {}, {}
    for kind in ["logit", "gbm"]:
        for name, cols in feature_sets.items():
            log(f"  {kind:5s} | {name:12s} | {len(cols)} features")
            P = oof_predictions(df[cols], y, kind, seeds, small_data=small_data)
            oof[(kind, name)] = P
            results[f"{kind}|{name}"] = {g: evaluate(y, P, m, n_boot) for g, m in subgroups.items()}
    base = list(feature_sets)[0]
    tests = {}
    for kind in ["logit", "gbm"]:
        for name in list(feature_sets)[1:]:
            for g, m in subgroups.items():
                mm = np.ones(len(y), bool) if m is None else m
                tests[f"{kind}|{name} vs {base}|{g}"] = delong_test(
                    y[mm], oof[(kind, name)][mm].mean(1), oof[(kind, base)][mm].mean(1))
    return results, tests, oof


def headline_table(results):
    rows = []
    for key, groups in results.items():
        kind, fs = key.split("|")
        for g, r in groups.items():
            rows.append({"learner": kind, "features": fs, "group": g, "n": r["n"],
                         "AUC": round(r["auc"], 3), "AUC 95% CI": f"{r['auc_ci95'][0]:.3f}-{r['auc_ci95'][1]:.3f}",
                         "AUC sd (seeds)": round(r["auc_sd_over_seeds"], 4), "PR-AUC": round(r["pr_auc"], 3),
                         "Brier": round(r["brier"], 4), "KS": round(r["ks"], 3),
                         "Default rate if 70% approved": round(r["approval_at_70pct"]["default_rate_approved"], 4)})
    return pd.DataFrame(rows)

print('LightGBM available:', HAVE_LGB)

In [ ]:
# Load Home Credit features and define the feature sets
p = f'{OUT}/home_credit_features.parquet'
hc = pd.read_parquet(p) if os.path.exists(p) else pd.read_csv(f'{OUT}/home_credit_features.csv.gz')
y_hc = hc['TARGET'].values
BUR = [c for c in hc.columns if c.startswith('bur_')]
ALT = [c for c in hc.columns if c.startswith('alt_')]
EXT = ['bur_ext_source_1', 'bur_ext_source_2', 'bur_ext_source_3']
FS_HC = {'bureau': BUR, 'alternative': ALT, 'combined': BUR + ALT}
thin = hc['thin_file'].values == 1
print(f'{len(hc):,} applicants; {thin.sum():,} thin-file ({thin.mean():.1%}); default rate {y_hc.mean():.2%}')
print('Bureau features:', BUR); print('Alternative features:', ALT)

In [ ]:
# Experiment 1: all applicants, evaluated overall and by group (about 25-35 minutes)
SEEDS = [11, 22, 33]
t0 = time.time()
res1, tests1, oof1 = run_suite(hc, y_hc, FS_HC, SEEDS,
                               {'all': None, 'thin_file': thin, 'with_bureau': ~thin})
print(f'Done in {(time.time()-t0)/60:.1f} min')
h1 = headline_table(res1); h1.insert(0, 'experiment', 'HC all applicants')
display(h1)
display(pd.DataFrame(tests1).T.round(4))

In [ ]:
# Experiment 2: trained and tested only on thin-file applicants (about 5 minutes)
hc_thin = hc[thin].reset_index(drop=True)
y_thin = hc_thin['TARGET'].values
FS_THIN = {'external_scores': EXT, 'alternative': ALT, 'combined': EXT + ALT}
t0 = time.time()
res2, tests2, oof2 = run_suite(hc_thin, y_thin, FS_THIN, SEEDS, {'thin_file': None})
print(f'Done in {(time.time()-t0)/60:.1f} min')
h2 = headline_table(res2); h2.insert(0, 'experiment', 'HC thin-file only')
display(h2); display(pd.DataFrame(tests2).T.round(4))

In [ ]:
# Experiment 3: Berka, real bank-account cash flow (about 3 minutes)
bk = pd.read_csv(f'{OUT}/berka_features.csv')
bk = bk[bk['enough_history'] == 1].reset_index(drop=True)
y_bk = bk['default'].values
CF = [c for c in bk.columns if c.startswith('cf_')]
TERMS = [c for c in bk.columns if c.startswith('terms_')]
FS_BK = {'loan_terms': TERMS, 'cashflow': CF, 'combined': TERMS + CF}
print(f'{len(bk)} loans, {y_bk.sum()} defaults')
t0 = time.time()
res3, tests3, oof3 = run_suite(bk, y_bk, FS_BK, list(range(20)), {'all': None}, small_data=True, n_boot=1000)
print(f'Done in {(time.time()-t0)/60:.1f} min')
h3 = headline_table(res3); h3.insert(0, 'experiment', 'Berka cash flow')
display(h3); display(pd.DataFrame(tests3).T.round(4))

In [ ]:
# Save results, plus out-of-fold predictions for Notebook 3
headline = pd.concat([h1, h2, h3], ignore_index=True)
headline.to_csv(f'{OUT}/headline_results.csv', index=False)
results = {'experiment_1_all_applicants': {'metrics': res1, 'delong_tests': tests1},
           'experiment_2_thin_file_only': {'metrics': res2, 'delong_tests': tests2},
           'experiment_3_berka': {'metrics': res3, 'delong_tests': tests3},
           'settings': {'seeds_hc': SEEDS, 'seeds_berka': 20, 'folds': 5, 'lightgbm': HAVE_LGB,
                        'features': {'hc_bureau': BUR, 'hc_alternative': ALT, 'berka_cashflow': CF, 'berka_terms': TERMS}}}
with open(f'{OUT}/model_results.json', 'w') as f:
    json.dump(results, f, indent=1, default=str)
np.savez_compressed(f'{OUT}/oof_predictions.npz',
                    **{f'exp1_{k}_{n}': v.mean(1) for (k, n), v in oof1.items()},
                    **{f'exp2_{k}_{n}': v.mean(1) for (k, n), v in oof2.items()},
                    **{f'exp3_{k}_{n}': v.mean(1) for (k, n), v in oof3.items()})
from google.colab import files
files.download(f'{OUT}/model_results.json'); files.download(f'{OUT}/headline_results.csv')
print('Saved. Send model_results.json and headline_results.csv to Claude.')